# Phase 1: Coordinate Regression & Hierarchical Localization

**Author:** Member 4 — Regression & Validation Lead  
**Objective:** Predict 2D coordinates (,  in metres) from Wi-Fi fingerprints, compare **global**, **per-building**, and **per-building-and-floor** regressors, evaluate oracle vs. end-to-end routing, and validate generalization across 5 USERID-grouped folds and the official held-out test set.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.load import load_data
from src.features.transform import FoldFeatureBuilder
from src.models.ml.regressors import CoordinateTargetScaler, HierarchicalCoordinateRegressor, make_base_regressor, compute_position_metrics

print("Libraries loaded successfully.")


## 1. Load Data & Target Scaling

Target coordinates ( and ) are continuous values in metres. To maintain gradient stability and distance weighting across axes, targets are standardized using training fold statistics only, and inverse-transformed prior to error calculation.


In [ ]:
train_df, test_df, meta = load_data()
coords = train_df[["LONGITUDE", "LATITUDE"]].to_numpy()

scaler = CoordinateTargetScaler()
scaled_coords = scaler.fit_transform(coords)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(coords[:, 0], coords[:, 1], s=2, alpha=0.3, color="#1f77b4")
axes[0].set_title("Original Coordinates (metres)")
axes[0].set_xlabel("Longitude (m)")
axes[0].set_ylabel("Latitude (m)")
axes[0].grid(True, linestyle=":", alpha=0.5)

axes[1].scatter(scaled_coords[:, 0], scaled_coords[:, 1], s=2, alpha=0.3, color="#ff7f0e")
axes[1].set_title("Standardized Coordinates (zero mean, unit variance)")
axes[1].set_xlabel("Standardized Longitude")
axes[1].set_ylabel("Standardized Latitude")
axes[1].grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()


## 2. Five-Fold USERID Grouped-CV Regression Results

Summary of Euclidean position errors (in metres) across 5 folds grouped by . We compare:
- **Global Regressor** (single model)
- **Per-Building Regressors** (Oracle vs. End-to-End with Member 3 classifiers)
- **Per-Building-and-Floor Regressors** (Oracle vs. End-to-End with Member 3 classifiers)


In [ ]:
summary_path = Path("results/tables/regressor_grouped_cv_summary.csv")
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display_cols = [
        "model", "routing", "routing_mode",
        "mean_error_mean", "mean_error_std",
        "median_error_mean", "p75_error_mean", "p95_error_mean",
        "mae_longitude_mean", "mae_latitude_mean"
    ]
    display(summary_df[display_cols].sort_values(["routing", "routing_mode", "mean_error_mean"]))
else:
    print("Results summary not found. Run scripts/train_regressors.py first.")


## 3. Official Held-Out Test Set Evaluation (Frozen Models)

Evaluation of the frozen models on the official held-out test set (, 1,111 samples).


In [ ]:
test_summary_path = Path("results/tables/regressor_test_summary.csv")
if test_summary_path.exists():
    test_df_summary = pd.read_csv(test_summary_path)
    display(test_df_summary[[
        "model", "routing", "routing_mode",
        "mean_error", "median_error", "p75_error", "p95_error",
        "mae_longitude", "mae_latitude"
    ]].sort_values(["routing", "routing_mode", "mean_error"]))
else:
    print("Test summary not found. Run scripts/evaluate_official_test.py first.")


## 4. Coordinate Error CDF (Cumulative Distribution Function)

The empirical CDF shows the share of test scans with localization error at or below each distance in metres.


In [ ]:
cdf_path = Path("results/figures/coordinate_error_cdf.png")
if cdf_path.exists():
    from IPython.display import Image
    display(Image(filename=str(cdf_path)))


## 5. Spatial Error Maps

Test coordinates coloured by Euclidean position error across individual building floor slabs.


In [ ]:
map_path = Path("results/figures/coordinate_error_maps.png")
if map_path.exists():
    from IPython.display import Image
    display(Image(filename=str(map_path)))


## 6. Live Single-Scan Demo (Build Guide Requirement)

Demonstrates end-to-end prediction of Building, Floor, and 2D Coordinates for a single incoming Wi-Fi fingerprint scan.


In [ ]:
# Pick a sample from the official validation/test set
sample_idx = 42
sample_row = test_df.iloc[[sample_idx]]

true_building = sample_row["BUILDINGID"].values[0]
true_floor = sample_row["FLOOR"].values[0]
true_coords = sample_row[["LONGITUDE", "LATITUDE"]].values[0]

print(f"--- Single Scan Demo (Sample #{sample_idx}) ---")
print(f"Ground Truth: Building {true_building}, Floor {true_floor}, Coords ({true_coords[0]:.2f}m, {true_coords[1]:.2f}m)")

# Load best trained model predictions from official test set
preds_file = Path("results/preds/regressors_test_preds.csv")
if preds_file.exists():
    p_df = pd.read_csv(preds_file)
    best_p = p_df[(p_df.model == "random_forest") & (p_df.routing == "per_building") & (p_df.routing_mode == "end_to_end")]
    if not best_p.empty:
        example = best_p.iloc[sample_idx]
        print(f"Predicted (End-to-End): Building {example['BUILDINGID_ROUTE']}, Floor {example['FLOOR_ROUTE']}")
        print(f"Predicted Coords: ({example['LONGITUDE_PRED']:.2f}m, {example['LATITUDE_PRED']:.2f}m)")
        print(f"Position Error: {example['ERROR_METRES']:.2f} metres")
